# M4 — Lexical hyperparameter search

**Цель.** Подобрать только параметры уже принятых lexical sources на frozen
`benchmark_aligned_proxy_v1`: stemmed all-field BM25, title char-TF-IDF и
весов их RRF. Основная метрика — macro **Recall@50**.

Эксперимент не использует benchmark labels и не меняет category partition.


## План

1. **Подготовка и протокол** — загрузить три исходных файла, восстановить
   group-disjoint proxy с seed 42 и начать live ClearML task.
2. **BM25 grid** — один stemmed Count index, затем подбор `k1 × b` без
   повторной токенизации корпуса.
3. **TF-IDF grid** — последовательная проверка небольшого набора char n-gram
   конфигураций, без параллельного хранения больших матриц.
4. **Fusion и решение** — подобрать RRF weights, сохранить все метрики и
   принять параметры только при измеренном gain относительно текущего M1.


In [ ]:
from __future__ import annotations

import ast
import gc
import json
import os
import platform
import random
import re
import shutil
import time
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow
import snowballstemmer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import GroupShuffleSplit

SEED = 42
TOP_K = 200
METRIC_KS = (50, 200)
BM25_BASELINE = {"k1": 1.5, "b": 0.75}
RRF_K = 60
TFIDF_BATCH_SIZE = 32
TOKEN_PATTERN = r"(?u)\b[0-9a-zа-я]{2,}\b"
NON_WORD_RE = re.compile(r"[^0-9a-zа-я]+")

REPO_ROOT = Path.cwd().resolve()
assert (REPO_ROOT / "pyproject.toml").exists(), (
    "Run from the repository root so .env and artifacts resolve correctly."
)
DATA_DIR = Path(os.environ.get("AVITO_DATA_DIR", "/Users/kite/Downloads/dataset"))
TRAIN_PATH = DATA_DIR / "train.parquet"
BENCHMARK_QUERIES_PATH = DATA_DIR / "benchmark_queries.parquet"
BENCHMARK_ITEMS_PATH = DATA_DIR / "benchmark_items.parquet"
ARTIFACT_DIR = REPO_ROOT / "artifacts/hpo/m4_lexical_hpo_s42"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

SEARCH_COLUMNS = [
    "search_query", "search_location_id", "search_is_delivery_search",
    "search_infm_params_text", "search_category",
]
ITEM_COLUMNS = [
    "item_id", "item_title_raw", "item_infm_params_text",
    "item_description_raw", "item_category_id",
]

random.seed(SEED)
np.random.seed(SEED)

def load_dotenv(path: Path) -> None:
    for raw_line in path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if line and not line.startswith("#") and "=" in line:
            key, value = line.split("=", 1)
            os.environ.setdefault(key.strip(), value.strip())

load_dotenv(REPO_ROOT / ".env")
required_env = ("CLEARML_API_ACCESS_KEY", "CLEARML_API_SECRET_KEY")
assert all(os.environ.get(key) for key in required_env), "Set ClearML credentials in .env."
assert all(path.exists() for path in (TRAIN_PATH, BENCHMARK_QUERIES_PATH, BENCHMARK_ITEMS_PATH))

print({
    "data_dir": str(DATA_DIR),
    "artifact_dir": str(ARTIFACT_DIR),
    "python": platform.python_version(),
    "pandas": pd.__version__,
    "pyarrow": pyarrow.__version__,
})


## 1. Frozen proxy and live experiment log

The split is identical to M0/M1: only train positives whose `item_id` appears
in `benchmark_items` are evaluated, and full normalized query contexts are
group-disjoint between train and validation.


In [ ]:
from clearml import Task

clearml_task = Task.init(
    project_name="avito-retrieval",
    task_name="M4__lexical_hpo__local__s42",
    reuse_last_task_id=False,
    auto_connect_arg_parser=False,
    auto_connect_frameworks={"detect_repository": False},
    auto_resource_monitoring=False,
    auto_connect_streams=False,
)
clearml_logger = clearml_task.get_logger()
clearml_task.connect(
    {
        "stage": "M4_lexical_hpo",
        "environment": "local",
        "validation_protocol": "benchmark_aligned_proxy_v1",
        "seed": SEED,
        "top_k": TOP_K,
        "bm25_grid": [
            {"k1": 1.0, "b": 0.50}, {"k1": 1.0, "b": 0.75},
            {"k1": 1.5, "b": 0.50}, {"k1": 1.5, "b": 0.75},
            {"k1": 1.5, "b": 0.90}, {"k1": 2.0, "b": 0.50},
            {"k1": 2.0, "b": 0.75}, {"k1": 2.0, "b": 0.90},
        ],
    },
    name="config",
)
print({"clearml_task_id": clearml_task.id, "offline_mode": False})


In [ ]:
load_started = time.perf_counter()
train_pairs = pd.read_parquet(TRAIN_PATH, columns=[*SEARCH_COLUMNS, "item_id"])
benchmark_queries = pd.read_parquet(
    BENCHMARK_QUERIES_PATH, columns=["query_id", *SEARCH_COLUMNS]
)
candidate_items = pd.read_parquet(BENCHMARK_ITEMS_PATH, columns=ITEM_COLUMNS).reset_index(drop=True)

def canonical_query_frame(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame[SEARCH_COLUMNS].copy()
    for column in ("search_query", "search_infm_params_text"):
        result[column] = (
            result[column].astype("string").fillna("<NA>").str.lower().str.strip()
            .str.replace(r"\s+", " ", regex=True)
        )
    for column in ("search_location_id", "search_is_delivery_search", "search_category"):
        result[column] = result[column].astype("string").fillna("<NA>")
    return result

all_contexts = pd.concat(
    [canonical_query_frame(train_pairs), canonical_query_frame(benchmark_queries)],
    ignore_index=True,
)
group_ids, _ = pd.factorize(pd.MultiIndex.from_frame(all_contexts), sort=False)
train_pairs["query_group"] = group_ids[: len(train_pairs)]

item_ids = candidate_items["item_id"].astype(str).to_numpy()
item_id_set = set(item_ids)
proxy_pairs = train_pairs.loc[train_pairs["item_id"].astype(str).isin(item_id_set)].copy()
splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
train_index, valid_index = next(splitter.split(proxy_pairs, groups=proxy_pairs["query_group"]))
proxy_valid_pairs = proxy_pairs.iloc[valid_index].copy()
validation_queries = (
    proxy_valid_pairs.sort_values("query_group").drop_duplicates("query_group")
    [["query_group", *SEARCH_COLUMNS]].reset_index(drop=True)
)
gold_by_group = proxy_valid_pairs.groupby("query_group", sort=False)["item_id"].agg(
    lambda values: frozenset(values.astype(str))
).to_dict()
gold_sets = [gold_by_group[group] for group in validation_queries["query_group"]]

category_to_indices = {
    str(category): group.index.to_numpy(dtype=np.int64)
    for category, group in candidate_items.groupby("item_category_id", sort=False)
}
all_indices = np.arange(len(candidate_items), dtype=np.int64)
allowed_indices_by_query = [
    category_to_indices.get(str(category), all_indices)
    for category in validation_queries["search_category"]
]

assert len(validation_queries) == 5310, "Frozen proxy changed unexpectedly."
assert len(validation_queries) == len(gold_sets)
print({
    "load_seconds": round(time.perf_counter() - load_started, 2),
    "candidate_items": len(candidate_items),
    "proxy_pairs": len(proxy_pairs),
    "validation_queries": len(validation_queries),
})


## 2. BM25 `k1 × b` search

All BM25 variants use the M1-selected Russian stemming and identical all-field
item text. The expensive token-count matrix is fitted once; only the Okapi
length normalization changes between configurations.


In [ ]:
STEMMER = snowballstemmer.stemmer("russian")

def normalize_russian_text(value: object) -> str:
    text = "" if pd.isna(value) else str(value)
    return " ".join(NON_WORD_RE.sub(" ", text.lower().replace("ё", "е")).split())

def stem_russian_text(value: object) -> str:
    return " ".join(STEMMER.stemWords(normalize_russian_text(value).split()))

def compose_document_text(frame: pd.DataFrame, fields: tuple[str, ...]) -> list[str]:
    text = frame[fields[0]].fillna("").astype(str)
    for field in fields[1:]:
        text = text.str.cat(frame[field].fillna("").astype(str), sep=" ")
    return text.tolist()

def top_k_from_scores(scores: np.ndarray, k: int) -> np.ndarray:
    k = min(k, len(scores))
    selected = np.argpartition(scores, len(scores) - k)[len(scores) - k:]
    return selected[np.argsort(scores[selected])[::-1]]

def recall_metrics(rankings: list[np.ndarray]) -> dict[str, float]:
    result: dict[str, float] = {}
    for k in METRIC_KS:
        recalls = [
            len(set(item_ids[ranking[:k]]) & gold) / len(gold)
            for ranking, gold in zip(rankings, gold_sets, strict=True)
        ]
        result[f"recall@{k}"] = float(np.mean(recalls))
    result["hit_rate@50"] = float(np.mean([
        bool(set(item_ids[ranking[:50]]) & gold)
        for ranking, gold in zip(rankings, gold_sets, strict=True)
    ]))
    return result

class ReusableBM25:
    """Exact sparse BM25; `k1` and `b` remain runtime parameters."""

    def fit(self, documents: list[str]) -> "ReusableBM25":
        self.vectorizer = CountVectorizer(
            preprocessor=stem_russian_text, token_pattern=TOKEN_PATTERN,
            lowercase=False, dtype=np.float32,
        )
        counts = self.vectorizer.fit_transform(documents)
        self.doc_len = np.asarray(counts.sum(axis=1)).ravel().astype(np.float32)
        self.n_docs = counts.shape[0]
        self.avgdl = float(self.doc_len.mean())
        self.matrix = counts.tocsc()
        del counts
        document_frequency = np.diff(self.matrix.indptr).astype(np.float64)
        idf = np.log((self.n_docs - document_frequency + 0.5) / (document_frequency + 0.5))
        average_idf = float(idf.mean())
        idf[idf < 0] = 0.25 * average_idf
        self.idf = idf.astype(np.float32)
        self.analyzer = self.vectorizer.build_analyzer()
        return self

    @property
    def n_features(self) -> int:
        return len(self.vectorizer.vocabulary_)

    def query_terms(self, query: str) -> list[tuple[int, int]]:
        term_counts = Counter(self.analyzer(query))
        return [
            (self.vectorizer.vocabulary_[token], int(query_tf))
            for token, query_tf in term_counts.items()
            if token in self.vectorizer.vocabulary_
        ]

    def top_k(self, term_features: list[tuple[int, int]], allowed: np.ndarray, *, k1: float, b: float) -> np.ndarray:
        norm = k1 * (1.0 - b + b * self.doc_len / self.avgdl)
        scores = np.zeros(self.n_docs, dtype=np.float32)
        for feature_idx, query_tf in term_features:
            start, stop = self.matrix.indptr[feature_idx : feature_idx + 2]
            rows = self.matrix.indices[start:stop]
            term_tf = self.matrix.data[start:stop]
            scores[rows] += query_tf * self.idf[feature_idx] * (
                term_tf * (k1 + 1.0) / (term_tf + norm[rows])
            )
        return allowed[top_k_from_scores(scores[allowed], TOP_K)]


In [ ]:
bm25_documents = compose_document_text(
    candidate_items, ("item_title_raw", "item_infm_params_text", "item_description_raw")
)
bm25_queries = validation_queries["search_query"].fillna("").astype(str).tolist()

bm25_fit_started = time.perf_counter()
bm25 = ReusableBM25().fit(bm25_documents)
bm25_fit_seconds = time.perf_counter() - bm25_fit_started
query_features = [bm25.query_terms(query) for query in bm25_queries]

BM25_SPECS = [
    {"k1": 1.0, "b": 0.50}, {"k1": 1.0, "b": 0.75},
    {"k1": 1.5, "b": 0.50}, {"k1": 1.5, "b": 0.75},
    {"k1": 1.5, "b": 0.90}, {"k1": 2.0, "b": 0.50},
    {"k1": 2.0, "b": 0.75}, {"k1": 2.0, "b": 0.90},
]

bm25_records: list[dict[str, object]] = []
best_bm25_rankings: list[np.ndarray] | None = None
best_bm25_record: dict[str, object] | None = None
for spec in BM25_SPECS:
    started = time.perf_counter()
    rankings = [
        bm25.top_k(terms, allowed, k1=spec["k1"], b=spec["b"])
        for terms, allowed in zip(query_features, allowed_indices_by_query, strict=True)
    ]
    metrics = recall_metrics(rankings)
    record = {
        "source": "bm25_stemmed_all_fields",
        **spec,
        **metrics,
        "index_seconds": bm25_fit_seconds,
        "retrieval_seconds": time.perf_counter() - started,
        "features": bm25.n_features,
    }
    bm25_records.append(record)
    if best_bm25_record is None or (
        record["recall@50"], record["recall@200"]
    ) > (best_bm25_record["recall@50"], best_bm25_record["recall@200"]):
        best_bm25_record, best_bm25_rankings = record, rankings
    print({key: record[key] for key in ("k1", "b", "recall@50", "recall@200", "retrieval_seconds")})

assert best_bm25_record is not None and best_bm25_rankings is not None
bm25_frame = pd.DataFrame(bm25_records).sort_values(
    ["recall@50", "recall@200"], ascending=False
).reset_index(drop=True)
display(bm25_frame)

# The lexical baseline present in M1 is recorded explicitly for a fair delta.
control_bm25 = next(record for record in bm25_records if record["k1"] == 1.5 and record["b"] == 0.75)
print({
    "bm25_fit_seconds": round(bm25_fit_seconds, 2),
    "control_recall@50": round(float(control_bm25["recall@50"]), 6),
    "best_bm25": {key: best_bm25_record[key] for key in ("k1", "b", "recall@50", "recall@200")},
})

del bm25, bm25_documents, query_features
gc.collect()


## 3. Title char-TF-IDF search

The final M1 source uses character n-grams on title text. Each candidate here
uses the same basic normalization, while `ngram_range`, `min_df` and feature
budget vary. Matrices are released between runs.


In [ ]:
TFIDF_SPECS = [
    {"name": "control_3_5_df2_200k", "ngram_range": (3, 5), "min_df": 2, "max_features": 200_000},
    {"name": "char_2_5_df2_200k", "ngram_range": (2, 5), "min_df": 2, "max_features": 200_000},
    {"name": "char_3_6_df2_200k", "ngram_range": (3, 6), "min_df": 2, "max_features": 200_000},
    {"name": "char_3_5_df3_200k", "ngram_range": (3, 5), "min_df": 3, "max_features": 200_000},
    {"name": "char_3_5_df2_250k", "ngram_range": (3, 5), "min_df": 2, "max_features": 250_000},
]

title_documents = candidate_items["item_title_raw"].fillna("").astype(str).tolist()
tfidf_queries = validation_queries["search_query"].fillna("").astype(str).tolist()
tfidf_records: list[dict[str, object]] = []
best_tfidf_rankings: list[np.ndarray] | None = None
best_tfidf_record: dict[str, object] | None = None

for spec in TFIDF_SPECS:
    fit_started = time.perf_counter()
    vectorizer = TfidfVectorizer(
        analyzer="char_wb",
        preprocessor=normalize_russian_text,
        lowercase=False,
        ngram_range=spec["ngram_range"],
        min_df=spec["min_df"],
        max_features=spec["max_features"],
        sublinear_tf=True,
        dtype=np.float32,
    )
    item_matrix = vectorizer.fit_transform(title_documents)
    index_seconds = time.perf_counter() - fit_started
    query_matrix = vectorizer.transform(tfidf_queries)

    retrieval_started = time.perf_counter()
    rankings: list[np.ndarray] = []
    for start in range(0, query_matrix.shape[0], TFIDF_BATCH_SIZE):
        stop = min(start + TFIDF_BATCH_SIZE, query_matrix.shape[0])
        scores_batch = (query_matrix[start:stop] @ item_matrix.T).toarray()
        for scores, allowed in zip(scores_batch, allowed_indices_by_query[start:stop], strict=True):
            rankings.append(allowed[top_k_from_scores(scores[allowed], TOP_K)])
    metrics = recall_metrics(rankings)
    record = {
        "source": "char_tfidf_title",
        **spec,
        **metrics,
        "index_seconds": index_seconds,
        "retrieval_seconds": time.perf_counter() - retrieval_started,
        "features": len(vectorizer.vocabulary_),
    }
    tfidf_records.append(record)
    if best_tfidf_record is None or (
        record["recall@50"], record["recall@200"]
    ) > (best_tfidf_record["recall@50"], best_tfidf_record["recall@200"]):
        best_tfidf_record, best_tfidf_rankings = record, rankings
    print({key: record[key] for key in ("name", "recall@50", "recall@200", "index_seconds", "retrieval_seconds")})
    del item_matrix, query_matrix, vectorizer
    gc.collect()

assert best_tfidf_record is not None and best_tfidf_rankings is not None
tfidf_frame = pd.DataFrame(tfidf_records).sort_values(
    ["recall@50", "recall@200"], ascending=False
).reset_index(drop=True)
display(tfidf_frame)
print({
    "best_tfidf": {key: best_tfidf_record[key] for key in ("name", "ngram_range", "min_df", "max_features", "recall@50", "recall@200")}
})


## 4. RRF weights, persisted result and decision

RRF receives only the best source representation from each grid. The control
is the M1 equal-weight fusion; a changed configuration is retained only if it
improves macro Recall@50 on the frozen proxy.


In [ ]:
def weighted_rrf(
    bm25_ranking: np.ndarray, tfidf_ranking: np.ndarray,
    *, bm25_weight: float, tfidf_weight: float,
) -> np.ndarray:
    scores: dict[int, float] = {}
    for weight, ranking in ((bm25_weight, bm25_ranking), (tfidf_weight, tfidf_ranking)):
        for rank, item_index in enumerate(ranking, start=1):
            item_index = int(item_index)
            scores[item_index] = scores.get(item_index, 0.0) + weight / (RRF_K + rank)
    ordered = sorted(scores, key=lambda item_index: (-scores[item_index], item_index))
    return np.asarray(ordered[:TOP_K], dtype=np.int64)

RRF_SPECS = [
    {"name": "equal_1_0_1_0", "bm25_weight": 1.0, "tfidf_weight": 1.0},
    {"name": "bm25_1_25_tfidf_1_0", "bm25_weight": 1.25, "tfidf_weight": 1.0},
    {"name": "bm25_1_5_tfidf_1_0", "bm25_weight": 1.5, "tfidf_weight": 1.0},
    {"name": "bm25_2_0_tfidf_1_0", "bm25_weight": 2.0, "tfidf_weight": 1.0},
    {"name": "bm25_1_0_tfidf_1_25", "bm25_weight": 1.0, "tfidf_weight": 1.25},
]
rrf_records: list[dict[str, object]] = []
best_rrf_rankings: list[np.ndarray] | None = None
best_rrf_record: dict[str, object] | None = None
for spec in RRF_SPECS:
    started = time.perf_counter()
    rankings = [
        weighted_rrf(bm25_rank, tfidf_rank, **{
            "bm25_weight": spec["bm25_weight"], "tfidf_weight": spec["tfidf_weight"]
        })
        for bm25_rank, tfidf_rank in zip(best_bm25_rankings, best_tfidf_rankings, strict=True)
    ]
    metrics = recall_metrics(rankings)
    record = {
        "source": "rrf_best_bm25_best_tfidf",
        **spec,
        **metrics,
        "index_seconds": 0.0,
        "retrieval_seconds": time.perf_counter() - started,
        "bm25_k1": best_bm25_record["k1"],
        "bm25_b": best_bm25_record["b"],
        "tfidf_name": best_tfidf_record["name"],
    }
    rrf_records.append(record)
    if best_rrf_record is None or (
        record["recall@50"], record["recall@200"]
    ) > (best_rrf_record["recall@50"], best_rrf_record["recall@200"]):
        best_rrf_record, best_rrf_rankings = record, rankings

assert best_rrf_record is not None and best_rrf_rankings is not None
rrf_frame = pd.DataFrame(rrf_records).sort_values(
    ["recall@50", "recall@200"], ascending=False
).reset_index(drop=True)
display(rrf_frame)

current_m1_recall_at_50 = 0.312988
all_results = pd.concat([
    bm25_frame.assign(experiment_type="bm25"),
    tfidf_frame.assign(experiment_type="tfidf"),
    rrf_frame.assign(experiment_type="rrf"),
], ignore_index=True, sort=False)
all_results.to_csv(ARTIFACT_DIR / "hpo_results.csv", index=False)

selected = {
    "validation_protocol": "benchmark_aligned_proxy_v1",
    "seed": SEED,
    "current_m1_recall_at_50": current_m1_recall_at_50,
    "best_bm25": best_bm25_record,
    "best_tfidf": best_tfidf_record,
    "best_rrf": best_rrf_record,
    "rrf_gain_vs_current_m1": float(best_rrf_record["recall@50"] - current_m1_recall_at_50),
    "accept_for_m1": bool(best_rrf_record["recall@50"] > current_m1_recall_at_50),
}
(ARTIFACT_DIR / "selected_config.json").write_text(
    json.dumps(selected, ensure_ascii=False, indent=2, default=str), encoding="utf-8"
)

rankings_frame = pd.DataFrame({
    "query_group": validation_queries["query_group"].astype(np.int64),
    "candidate_item_ids_top200": [" ".join(item_ids[row]) for row in best_rrf_rankings],
})
rankings_frame.to_parquet(ARTIFACT_DIR / "validation_best_rrf_top200.parquet", index=False)

manifest = {
    "stage": "M4_lexical_hpo",
    "seed": SEED,
    "validation_queries": len(validation_queries),
    "candidate_items": len(candidate_items),
    "source_files": {
        path.name: {"bytes": path.stat().st_size, "modified_ns": path.stat().st_mtime_ns}
        for path in (TRAIN_PATH, BENCHMARK_QUERIES_PATH, BENCHMARK_ITEMS_PATH)
    },
    "selected": selected,
}
(ARTIFACT_DIR / "manifest.json").write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2, default=str), encoding="utf-8"
)

for _, record in all_results.iterrows():
    clearml_logger.report_scalar(
        "Recall@50", str(record.get("name", f'{record["source"]}_{record.get("k1", "")}')),
        float(record["recall@50"]), 0
    )
clearml_logger.report_table("M4 lexical HPO", "all_results", 0, table_plot=all_results)
clearml_task.set_parameter("results/best_bm25_recall_at_50", float(best_bm25_record["recall@50"]))
clearml_task.set_parameter("results/best_tfidf_recall_at_50", float(best_tfidf_record["recall@50"]))
clearml_task.set_parameter("results/best_rrf_recall_at_50", float(best_rrf_record["recall@50"]))
clearml_task.set_parameter("results/rrf_gain_vs_current_m1", selected["rrf_gain_vs_current_m1"])
for artifact_name, artifact_path in {
    "m4_hpo_results": ARTIFACT_DIR / "hpo_results.csv",
    "m4_selected_config": ARTIFACT_DIR / "selected_config.json",
    "m4_manifest": ARTIFACT_DIR / "manifest.json",
}.items():
    clearml_task.upload_artifact(artifact_name, artifact_object=artifact_path)
clearml_task.close()

print({
    "best_bm25": {key: best_bm25_record[key] for key in ("k1", "b", "recall@50")},
    "best_tfidf": {key: best_tfidf_record[key] for key in ("name", "recall@50")},
    "best_rrf": {key: best_rrf_record[key] for key in ("name", "recall@50")},
    "gain_vs_current_m1": round(selected["rrf_gain_vs_current_m1"], 6),
    "accept_for_m1": selected["accept_for_m1"],
    "artifact_dir": str(ARTIFACT_DIR),
})


## Следующее решение

- Если `accept_for_m1 = true`, перенести только выбранные BM25, TF-IDF и RRF
  параметры в M1/M9 и повторно собрать финальный `answer.csv`.
- Если прироста нет, сохранить текущую M1 configuration.
- Квоты M2/E5 не меняются здесь: это отдельный M6 fusion HPO, чтобы не
  смешивать источники и интерпретацию результата.
